## Введение 

### Настройка окружения
Т.к. библиотека довольно долгое время не получает обновлений, используем изолированную среду с python3.6  

```sh
conda create -n lightfm python=3.6
conda activate lightfm
conda install -c conda-forge lightfm pandas scikit-learn
```

### Загрузка данных 

```sh
wget https://storage.yandexcloud.net/labintsev/data-science-for-good-careervillage.zip
unzip data-science-for-good-careervillage.zip -d data/careervillage
rm data-science-for-good-careervillage.zip
``` 

**Обзор ноутбука:**  
Этот ноутбук содержит решение для соревнования по рекомендательным системам CareerVillage. CareerVillage - это сайт, на котором студенты могут задавать вопросы профессионалам, преимущественно по вопросам построения карьеры. 
В ноутбуке приведен пример гибридной рекомендательной системы для рекомендации вопросов студентов профессионалам на CareerVillage.org. Рекомендательная система сопоставляет профессионалов с вопросами по тегам, на которые они подписаны, тегам вопросов, на которые они уже отвечали, а также по похожим тегам. Кроме того, она помогает справляться с одной из наиболее серьезных проблем рекомендательных систем CareerVillage, включая проблему холодного старта и другие.


**Постановка задачи соревнования:**  
CareerVillage.org — это некоммерческая организация, помогающая молодежи с ограниченным доступом к возможностям получать информацию для построения своей карьеры. Студенты могут задавать вопросы на CareerVillage.org, а профессионалы (эксперты, которым нравится помогать студентам) отвечают на них. Сложность заключается в том, что CareerVillage необходимо рекомендовать профессионалам подходящие вопросы так, чтобы они соответствовали их интересам. Это увеличивает вероятность того, что вопрос получит ответ. Поэтому в рамках этого соревнования требуется построить рекомендательную систему, которая будет корректно рекомендовать вопросы в соответствии с интересами профессионалов.

## Рекомендательные системы
Прежде чем углубляться в решение, давайте сначала разберемся с основной терминологией рекомендательных систем. По определению, рекомендательная система — это система, которая находит и предлагает пользователям подходящий контент или цифровые объекты на основе их интересов. Рекомендательные системы стали важной частью современных сайтов, таких как Amazon, Netflix или Flickr. Использование эффективных рекомендательных систем может повышать кликабельность, выручку и другие показатели успеха.

Сложность состоит в том, чтобы находить релевантные объекты, даже если в целом они непопулярны. Рекомендательные системы используют доступный контекст — например, информацию о пользователе, времени, местоположении и т.д. — чтобы отфильтровывать подходящие элементы. Благодаря этому можно успешно рекомендовать даже объекты из «длинного хвоста» распределения популярности. [1]


**Типы рекомендательных систем:**
1. **Коллаборативная фильтрация:** Коллаборативная фильтрация — это метод автоматического прогнозирования интересов пользователя на основе предпочтений или вкусов большого числа других пользователей. Основная идея этого подхода состоит в том, что если человек A имеет такое же мнение, как человек B, по одному вопросу, то с высокой вероятностью A будет разделять мнение B и по другому вопросу, по сравнению со случайно выбранным человеком. Ниже показано, как работает коллаборативная фильтрация [5].

![](https://i.imgur.com/FZli7DC.gif)  

Однако у коллаборативной фильтрации есть одна серьезная проблема — **«холодный старт»**. Как мы уже видели, коллаборативная фильтрация может быть мощным способом рекомендаций на основе истории пользователя, но что делать, если такой истории нет? Это и называется проблемой холодного старта, и она относится как к новым объектам, так и к новым пользователям. Объекты с богатой историей рекомендуются часто, а те, у которых истории нет, никогда не попадают в механизм рекомендаций, что создает положительную петлю обратной связи. В то же время у новых пользователей нет истории, поэтому система не может дать им качественные рекомендации. Возможное решение: на этапе онбординга можно собирать базовую информацию о пользователе или импортировать контакты из социальных сетей, чтобы быстрее понять его предпочтения [4].

2. **Контентная фильтрация:** Эти методы основаны на описании объекта и профиле предпочтений пользователя. В контентно-ориентированной рекомендательной системе для описания объектов используются ключевые слова, а также строится профиль пользователя, который отражает, какие типы объектов ему нравятся. Иными словами, алгоритмы пытаются рекомендовать продукты, похожие на те, которые пользователь любил в прошлом. Идея проста: если вам нравится некоторый объект, то вам, вероятно, понравится и «похожий» объект. Например, это часто применяется при рекомендации фильмов или музыки. [2]

    Одна из основных проблем этого подхода — **разнообразие**. Релевантность важна, но этого недостаточно. Если вам понравились фильмы *Star Wars*, велика вероятность, что вам понравится и *The Empire Strikes Back*, но вряд ли вам нужен рекомендательный движок, чтобы сказать это. Важно также, чтобы рекомендательная система выдавала новые и неожиданные результаты, а также обеспечивала разнообразие, то есть охватывала широкий спектр интересов пользователя. [3]

3. **Гибридная рекомендательная система:** Гибридная рекомендательная система — это особый тип системы, который сочетает в себе методы контентной и коллаборативной фильтрации. В некоторых случаях такое объединение оказывается более эффективным. Гибридные подходы можно реализовать разными способами: отдельно строить прогнозы на основе контента и коллаборации, а затем объединять их; либо добавлять контентные возможности к коллаборативному подходу и наоборот. Несколько исследований эмпирически сравнивают эффективность гибридных методов с чисто коллаборативными и контентными и показывают, что гибридные методы могут давать более точные рекомендации. Они также помогают преодолевать типичные проблемы рекомендательных систем, такие как холодный старт и разреженность данных.


**Типы данных для построения рекомендательных систем:** Существует два основных типа данных, доступных для построения рекомендательной системы.
1. **Явная обратная связь (Explicit feedback):** Это данные о явной реакции пользователя на продукт, например оценки или рейтинги. Они напрямую показывают, нравится пользователю продукт или нет.

2. **Неявная обратная связь (Implicit feedback):** В этом случае у нас нет прямых данных о том, как пользователь оценивает продукт. Примеры неявной обратной связи — клики, просмотренные фильмы, прослушанные песни, покупки или назначенные теги.

Теперь мы понимаем, что такое рекомендательная система и какие бывают ее типы. Далее мы рассмотрим, какой метод лучше всего подходит для построения рекомендательной системы для CareerVillage.org.

##  **Выбор рекомендательной системы для CareerVillage**

Нам нужно построить рекомендательную систему для CareerVillage. Давайте сначала разберёмся, какие данные предоставляет CareerVillage.

CareerVillage даёт нам набор данных о:
* профессионалах;
* вопросах, на которые ответили профессионалы;
* студентах;
* вопросах студентов.

Сначала определим, какие данные у нас есть для построения рекомендательной системы: это явная или неявная обратная связь?

Поскольку у нас нет оценок или эквивалентной матрицы, позволяющей определить, насколько профессионалу нравится тот или иной вопрос, мы работаем с данными **неявной обратной связи**.

Примеры неявной обратной связи на CareerVillage:
* теги вопросов, на которые пользователи дали ответы;
* местоположение.

**Какой метод выбрать?**

1. **Коллаборативная фильтрация** нам не подходит: она приведёт к проблеме «холодного старта». Значительная часть профессионалов пока не ответила ни на один вопрос, а для новых профессионалов ситуация будет ещё сложнее.

2. **Контентная фильтрация** тоже вызовет трудности:
   * не получится обеспечить достаточное разнообразие рекомендаций;
   * у большинства профессионалов нет чёткой информации, необходимой для построения матрицы пользовательских характеристик (user feature matrix).

**Какое решение?**

Оптимальный вариант — использовать **гибридную модель**. Она:
* объединит методы коллаборативной и контентной фильтрации;
* позволит формировать надёжные рекомендации за счёт анализа:
    * тегов профессионалов (если они есть);
    * интересов схожих профессионалов.

Такой подход поможет решить сразу две проблемы:
* проблему «холодного старта»;
* проблему недостатка разнообразия в рекомендациях.

Таким образом, гибридная модель идеально подходит для решения поставленной задачи.

## Гибридная рекомендательная система LightFM для CareerVillage

Гибридная рекомендательная система — это особый тип системы, использующий одновременно коллаборативную и контентную фильтрацию для формирования рекомендаций. Это делает гибридный подход очень эффективным методом построения рекомендательных систем. В данном проекте мы будем использовать **модель LightFM** — гибридную модель матричной факторизации от компании Lyst.

#### 1. Что такое LightFM?

**LightFM** — это гибридная модель матричной факторизации, в которой пользователи и элементы (items) представляются как линейные комбинации латентных факторов их контентных характеристик.

Ключевые преимущества LightFM:

* превосходит коллаборативные и контентные модели в сценариях **«холодного старта»** или при **разреженных данных** о взаимодействиях (за счёт использования метаданных о пользователях и элементах);
* показывает как минимум такую же эффективность, как чисто коллаборативная модель матричной факторизации, когда данных о взаимодействиях достаточно.

В LightFM:
* как в коллаборативной фильтрации, пользователи и элементы представляются в виде **латентных векторов** (эмбеддингов);
* как в контентной модели, эти векторы полностью определяются функциями (в данном случае линейными комбинациями) эмбеддингов характеристик контента, описывающих каждого пользователя или элемент.

**Пример:** если фильм «Волшебник страны Оз» описывается следующими характеристиками: «музыкальное фэнтези», «Джуди Гарленд» и «Волшебник страны Оз», то его латентное представление будет задаваться суммой латентных представлений этих характеристик. Таким образом, LightFM объединяет преимущества контентных и коллаборативных рекомендательных систем.

#### 2. Как работает LightFM

Модель LightFM обучается создавать **эмбеддинги** (латентные представления в многомерном пространстве) для пользователей и элементов таким образом, чтобы закодировать предпочтения пользователей относительно элементов.

При перемножении эти представления дают оценки для каждого элемента для заданного пользователя: элементы с высокими оценками с большей вероятностью будут интересны пользователю.

Представления пользователей и элементов выражаются через представления их характеристик:
* для каждой характеристики оценивается свой эмбеддинг;
* затем эти характеристики суммируются, чтобы получить представления для пользователей и элементов.

Математическое описание:

1. Латентное представление пользователя $u$ задаётся суммой латентных векторов его характеристик:
$$
q_{u} = \sum_{j \in f_{u}} e_{j},
$$
где:
* $q_{u}$ — латентный вектор пользователя $u$;
* $f_{u}$ — множество характеристик пользователя $u$;
* $e_{j}$ — латентный вектор характеристики $j$.

2. Аналогично для элементов:
$$
p_{i} = \sum_{j \in f_{i}} e_{j},
$$
где:
* $p_{i}$ — латентный вектор элемента $i$;
* $f_{i}$ — множество характеристик элемента $i$.

3. Прогноз модели для пользователя $u$ и элемента $i$ задаётся скалярным произведением представлений пользователя и элемента, скорректированным на смещения характеристик пользователя и элемента:
$$
\hat{r}_{ui} = f \left( q_{u} \cdot p_{i} + b_{u} + b_{i} \right),
$$
где:
* $\hat{r}_{ui}$ — прогнозируемая оценка взаимодействия пользователя $u$ с элементом $i$;
* $b_{u}$ — смещение пользователя $u$;
* $b_{i}$ — смещение элемента $i$;
* $f(\cdot)$ — функция активации (например, сигмоида), преобразующая результат в вероятность или оценку.

#### 3. Почему LightFM подходит для CareerVillage?

Выбор LightFM для рекомендательной системы CareerVillage обоснован следующими преимуществами:

1. **Эффективность в условиях «холодного старта» и разреженных данных.** LightFM показывает как минимум такую же эффективность, как чисто контентные модели, и существенно превосходит их, когда:
    * в обучающем наборе доступны коллаборативные данные;
    * модель включает характеристики пользователей.

   Это критически важно для CareerVillage, где постоянно появляются новые вопросы и студенты, создавая типичные условия для проблемы «холодного старта».

2. **Эффективность при обилии коллаборативных данных.** Когда данных достаточно (тёплый старт, плотная матрица «пользователь‑элемент»), LightFM работает как минимум так же хорошо, как модель матричной факторизации (MF).

3. **Семантическая ценность эмбеддингов.** Эмбеддинги, создаваемые LightFM, кодируют важную семантическую информацию о характеристиках и могут использоваться для смежных задач рекомендаций — например, для рекомендаций тегов. Это особенно важно для CareerVillage: модель сможет находить схожие теги и рекомендовать профессионалам вопросы с тегами, соответствующими их интересам.

Таким образом, LightFM идеально подходит для CareerVillage благодаря своей гибкости, способности работать с разными типами данных и эффективности в разнообразных сценариях — от «холодного старта» до ситуаций с обилием взаимодействий.

### Библиотека LightFM для Python

Существует удобная библиотека для построения моделей LightFM — она разработана компанией Lyst и активно поддерживается сообществом. Библиотека имеет более 2 400 звёзд на GitHub и зарекомендовала себя в продакшене у таких компаний, как Lyst и Sketchfab.


#### Преимущества библиотеки LightFM

1. **Реализация WARP‑потери (WARP loss)** для ранжирования с неявной обратной связью. Это ключевой плюс для CareerVillage, поскольку наши данные — именно неявная обратная связь.
2. **Высокая производительность:** библиотека написана на Cython и распараллелена через HOGWILD SGD, что обеспечивает скорость работы выше, чем у большинства самописных реализаций.
3. **Проверена на практике:** используется в продакшене многими известными брендами.
4. **Дружественный API:** простой и понятный интерфейс упрощает построение модели.
5. **Метрики оценки:** библиотека предоставляет инструменты для оценки качества модели.
6. **Скорость:** быстрая работа на больших наборах данных.

#### Что такое WARP (Weighted Approximate‑Rank Pairwise)

WARP — специальный метод оптимизации для задач ранжирования при неявной обратной связи. Его принцип работы:

1. Для заданной пары «пользователь — позитивный элемент» случайно выбирается негативный элемент из всех остальных.
2. Модель вычисляет предсказания для обоих элементов.
3. Если предсказание для негативного элемента превышает предсказание для позитивного (плюс некоторый запас), выполняется градиентное обновление: модель корректируется так, чтобы повысить ранг позитивного элемента и понизить ранг негативного.
4. Если нарушения ранга нет, продолжается выборка негативных элементов до тех пор, пока не будет найдено нарушение.
5. Величина обновления градиента зависит от того, сколько попыток потребовалось для нахождения нарушения:
    * если нарушение найдено с первой попытки — выполняется большое обновление (значит, много негативных элементов ранжируются выше позитивных, и модель нужно сильно скорректировать);
    * если потребовалось много выборок — выполняется небольшое обновление (модель близка к оптимуму и требует лишь тонкой настройки).

#### Как работает библиотека LightFM: пошаговое руководство

**Шаг 1. Подготовка данных и создание датасета LightFM**

Используем API библиотеки для преобразования исходных данных в формат, понятный LightFM.

**Шаг 2. Построение матрицы взаимодействий и признаков пользователей/элементов**

Создаём:
* матрицу взаимодействий (user‑item interaction matrix) — отражает неявную обратную связь (ответы на вопросы, просмотры и т. д.);
* матрицы признаков пользователей (user features) — например, профессия, местоположение, теги интересов;
* матрицы признаков элементов (item features) — теги вопросов, тематика, уровень сложности и т. д.

**Шаг 3. Создание и обучение модели**

1. Инициализируем модель LightFM, выбирая тип функции потерь (например, `loss='warp'` для WARP).
2. Обучаем модель на подготовленных данных:
   * передаём матрицу взаимодействий;
   * при необходимости передаём признаки пользователей и элементов.

Пример кода:
```python
from lightfm import LightFM

# Инициализация модели с WARP-потерей
model = LightFM(loss='warp')

# Обучение модели
model.fit(interactions, user_features=user_features, item_features=item_features, epochs=30, num_threads=2)
```

**Шаг 4. Оценка модели**

Библиотека предоставляет метрики для оценки качества рекомендаций, например:
* Precision@k — доля релевантных рекомендаций среди топ‑k;
* Recall@k — доля найденных релевантных элементов среди всех релевантных;
* AUC — площадь под ROC‑кривой (общая способность модели ранжировать).

Пример оценки:
```python
from lightfm.evaluation import precision_at_k

# Оценка Precision@k
test_precision = precision_at_k(model, test_interactions, k=10).mean()
print(f'Precision@10: {test_precision:.4f}')
```

**Шаг 5. Формирование предсказаний**

После обучения модель может выдавать рекомендации для пользователей:
```python
# Получение рекомендаций для пользователя с ID=0
scores = model.predict(0, np.arange(n_items))
# Топ-10 рекомендаций
top_items = np.argsort(-scores)[:10]
```

---

#### Почему LightFM подходит для CareerVillage?

1. **Неявная обратная связь.** WARP оптимизирован для сценариев с неявной обратной связью (клики, ответы, просмотры), что идеально соответствует данным CareerVillage.
2. **Холодный старт.** Использование признаков пользователей и элементов помогает давать осмысленные рекомендации даже для новых профессионалов или вопросов.
3. **Разнообразие.** Модель учитывает как коллаборативные паттерны, так и контентные характеристики (теги, профили), что повышает разнообразие рекомендаций.
4. **Скорость и масштабируемость.** Библиотека быстро обрабатывает большие наборы данных, что критично для растущей платформы.

---

**Где узнать больше?**

Официальная документация библиотеки LightFM: https://lyst.github.io/lightfm/docs/index.html

### Эффективность решения для рекомендательной системы CareerVillage

Разберу, почему предложенное решение на базе LightFM эффективно решает ключевые задачи CareerVillage.

#### 1. Решение основных проблем рекомендательной системы

**Проблема «холодного старта»**

LightFM отлично справляется с «холодным стартом» благодаря гибридной природе:
* использует признаки пользователей и элементов для рекомендаций;
* при отсутствии взаимодействий (для новых пользователей/вопросов) переключается на контентный или коллаборативный метод — в зависимости от доступных данных;
* учитывает метаданные (теги, профили) даже при отсутствии истории взаимодействий.

**Ранжирование вопросов**

Цель — обеспечить быстрый ответ на вопросы студентов. Для этого модель учитывает количество ответов на вопрос:
* вопросы с большим числом ответов (например, 10+) менее приоритетны — вероятно, студент уже получил достаточно информации;
* вопросы с малым числом ответов (менее 3) требуют срочных рекомендаций.

Решениение: взвешивание взаимодействий в матрице. Вес рассчитывается как $\frac{1}{\text{num\_of\_ans\_per\_ques}}$. Этот вес передаётся в модель вместе с матрицей взаимодействий, что повышает приоритет вопросов с малым числом ответов.

Пример кода:
```python
import numpy as np

# Предположим, у нас есть массив с количеством ответов на каждый вопрос
num_answers_per_question = np.array([10, 2, 5, 1, 8])

# Расчёт весов
weights = 1.0 / num_answers_per_question

# Нормализация весов (опционально)
weights = weights / np.max(weights)

print("Веса для вопросов:", weights)
# Вывод: [0.1  0.5  0.2  1.   0.125]
```

**Схожесть тегов**

Профессионалы и вопросы могут иметь несколько тегов, часто похожих, но не идентичных (например, «data science» и «data analytics»). LightFM решает эту проблему:
* создаёт эмбеддинги признаков пользователей и элементов;
* кодирует семантическую информацию о тегах;
* находит схожесть между тегами, даже если они немного отличаются.

Это позволяет:
* рекомендовать профессионалам вопросы с похожими тегами;
* учитывать нюансы в формулировках тегов;
* повышать релевантность рекомендаций.

**Возможность добавления новых признаков**

Архитектура решения спроектирована так, чтобы легко добавлять новые признаки:
* предусмотрены классы и функции для построения датасета и модели;
* модульный код позволяет тестировать новые признаки без перестройки всей системы;
* CareerVillage может экспериментировать с новыми данными (например, добавить стаж работы, уровень сложности вопроса и т. д.).

**Скорость работы**

Решениеение обеспечивает высокую скорость обучения и предсказаний благодаря:
* использованию библиотеки LightFM (оптимизирована на Cython);
* применению WARP‑потери (эффективна для неявной обратной связи);
* распараллеливанию вычислений (HOGWILD SGD).

#### 2. Готовность к развёртыванию в продакшене

Решениеение готово к внедрению в продакшен:
* **Пошаговые инструкции:** предоставлены чёткие руководства по развёртыванию модели.
* **Переиспользуемые функции:** созданы универсальные функции для типовых сценариев, упрощающие добавление новых функций.
* **Форматированный и документированный код:** все методы, функции и классы снабжены комментариями, что облегчает понимание и поддержку кода.

#### 3. Хорошая документация

Решение полностью документировано:
* **Полное описание:** от основ рекомендательных систем до объяснения работы модели и шагов развёртывания.
* **Комментарии к коду:** большинство строк кода снабжены пояснениями для лёгкого понимания.
* **Блок‑схемы:** предоставлены визуальные схемы структуры модели и блокнота, облегчающие восприятие архитектуры.

---

### Итог

Предложенное решение на базе LightFM:
* эффективно решает ключевые проблемы CareerVillage (холодный старт, ранжирование вопросов, схожесть тегов);
* гибко адаптируется к новым данным и признакам;
* обеспечивает высокую скорость работы;
* готово к развёртыванию в продакшене;
* полностью документировано и понятно для команды разработки.

Теперь можно переходить к построению модели. Давайте начнём!


### Обзор ядра (Kernel Overview)

Ядро решения структурировано в три основных раздела. Разберу каждый подробно — с пояснением целей, содержания и логики разделения.

#### 1. Обзор решения (Solutions Overview)

**Цель:** дать целостное понимание задачи, выбранного подхода и его преимуществ.

**Содержание раздела:**
* базовые понятия рекомендательных систем: определение, назначение, ключевые задачи;
* обзор типов рекомендательных систем: коллаборативная фильтрация, контентная фильтрация, гибридные системы;
* обоснование выбора LightFM:
    * гибридная природа модели (объединение коллаборативных и контентных методов);
    * эффективность при «холодном старте» и разреженных данных;
    * поддержка неявной обратной связи через WARP‑потерю;
* демонстрация эффективности решения для CareerVillage:
    * решение проблемы «холодного старта»;
    * ранжирование вопросов по срочности (веса на основе числа ответов);
    * учёт схожести тегов через семантические эмбеддинги;
    * гибкость для добавления новых признаков;
    * высокая скорость обучения и предсказаний;
* подтверждение готовности к продакшену: документация, переиспользуемый код, инструкции по развёртыванию.

**Результат:** читатель получает ясное представление о том, *почему* выбран LightFM и *как* он решает специфические задачи CareerVillage.

#### 2. Построение модели (Model Building)

**Цель:** пошагово реализовать модель с подробными пояснениями каждого действия.

**Подход:** итеративное построение с акцентом на понимание логики и причин выбора конкретных решений.

**Ключевые шаги (с документацией на каждом этапе):**

1. **Подготовка данных:**
    * загрузка и очистка исходных данных (вопросы, ответы, профили пользователей, теги);
    * формирование матрицы взаимодействий (user‑item) на основе неявной обратной связи (ответы на вопросы);
    * создание матриц признаков пользователей (user features) и вопросов (item features) — теги, местоположение, тематика и т. д.;
    * расчёт весов для вопросов на основе числа ответов: $\text{вес} = \frac{1}{\text{num\_of\_ans\_per\_ques}}$.

2. **Инициализация и обучение модели:**
    * выбор гиперпараметров (тип функции потерь — `loss='warp'`, число эпох, число потоков);
    * инициализация модели LightFM;
    * обучение на подготовленных данных (матрица взаимодействий + признаки).

3. **Оценка модели:**
    * разделение данных на обучающую и тестовую выборки;
    * расчёт метрик качества: Precision@k, Recall@k, AUC;
    * анализ результатов и подбор оптимальных гиперпараметров.

4. **Формирование рекомендаций:**
    * предсказание оценок для всех пар «пользователь‑вопрос»;
    * ранжирование вопросов для каждого профессионала;
    * отбор топ‑N рекомендаций.

5. **Анализ результатов:**
    * визуализация метрик (графики Precision@k, ROC‑кривые);
    * разбор примеров рекомендаций (успешных и неудачных);
    * проверка решения задач: «холодный старт», ранжирование по срочности, учёт тегов.

**Визуальное сопровождение:** блок‑схема последовательности шагов 

![](https://i.imgur.com/UyaNRZg.png)

**Результат:** полностью обученная и протестированная модель с пониманием того, *как* и *почему* каждый шаг влияет на итоговую рекомендацию.

#### 3. Модель в продакшене (Model in Production)

**Цель:** преобразовать исследовательский код в готовую к развёртыванию систему.

**Отличие от шага 2:** в шаге 2 код писался для понимания и экспериментов (линейный скрипт, жёстко заданные параметры). В этом разделе код структурируется в переиспользуемые компоненты для продакшена.

**Ключевые действия:**

1. **Создание классов для каждого этапа:**
    * `DataProcessor` — загрузка, очистка, формирование матриц;
    * `ModelTrainer` — инициализация, обучение, сохранение модели;
    * `Evaluator` — расчёт метрик, визуализация результатов;
    * `Recommender` — генерация топ‑N рекомендаций для пользователя.

2. **Построение пайплайна:**
    * объединение классов в единый конвейер обработки;
    * автоматизация шагов: загрузка данных → обучение → оценка → сохранение модели;
    * настройка периодического переобучения (например, раз в сутки).

3. **Интеграция с инфраструктурой CareerVillage:**
    * API для получения рекомендаций (REST/gRPC);
    * подключение к базе данных пользователей и вопросов;
    * логирование и мониторинг (метрики качества, время ответа, ошибки).

4. **Оптимизация для продакшена:**
    * кэширование часто запрашиваемых рекомендаций;
    * параллельная обработка запросов;
    * обработка ошибок и fallback‑стратегии (например, при «холодном старте»).

**Визуальное сопровождение:** схема развёртывания модели (см. изображение по ссылке

![](https://i.imgur.com/7fG6gvc.png)
 
 Это не жёсткий шаблон, а иллюстрация логики интеграции.

**Результат:** готовый к развёртыванию модуль с:
* чёткой архитектурой (классы, интерфейсы);
* автоматизированным пайплайном;
* интеграцией с существующей инфраструктурой;
* механизмами мониторинга и поддержки.

### Итог

Разделение на три раздела обеспечивает:
* **Понимание:** шаг 2 даёт глубокое понимание модели и данных.
* **Гибкость:** код шага 2 легко модифицировать для экспериментов.
* **Готовность к продакшену:** шаг 3 превращает исследовательский код в промышленный продукт.

Теперь можно переходить к **шагу 2 (Построение модели)** — реализации каждого этапа с документацией и пояснениями.

# Gathering Data 
CareerVillage provides us a very rich set of datasets for this competition. Dataset contains information about students, professionals, questions, answers, comment and specially tags. This competition already contains lot of great EDA and data analysis on this data. Feel free to look at those. This will give you very rich idea about the dataset.

In [30]:
################################################
# Importing necessary library
################################################
import numpy as np
import pandas as pd

# all lightfm imports 
from lightfm.data import Dataset
from lightfm import LightFM
from lightfm import cross_validation
from lightfm.evaluation import precision_at_k
from lightfm.evaluation import auc_score

# imports re for text cleaning 
import re
from datetime import datetime, timedelta

# we will ignore pandas warning 
import warnings
warnings.filterwarnings('ignore')


In [4]:
############################################
# Read all our datasets and store them in pandas dataframe objects. 
############################################
base_path = 'data/careervillage/'
df_answer_scores = pd.read_csv(
    base_path + 'answer_scores.csv')

df_answers = pd.read_csv(
    base_path + 'answers.csv',
    parse_dates=['answers_date_added'])

df_comments = pd.read_csv(
    base_path + 'comments.csv')

df_emails = pd.read_csv(
    base_path + 'emails.csv')

df_group_memberships = pd.read_csv(
    base_path + 'group_memberships.csv')

df_groups = pd.read_csv(
    base_path + 'groups.csv')

df_matches = pd.read_csv(
    base_path + 'matches.csv')

df_professionals = pd.read_csv(
    base_path + 'professionals.csv',
    parse_dates=['professionals_date_joined'])

df_question_scores = pd.read_csv(
    base_path + 'question_scores.csv')

df_questions = pd.read_csv(
    base_path + 'questions.csv',
    parse_dates=['questions_date_added'])

df_school_memberships = pd.read_csv(
    base_path + 'school_memberships.csv')

df_students = pd.read_csv(
    base_path + 'students.csv',
    parse_dates=['students_date_joined'])

df_tag_questions = pd.read_csv(
    base_path + 'tag_questions.csv')

df_tag_users = pd.read_csv(
    base_path + 'tag_users.csv')

df_tags = pd.read_csv(
    base_path + 'tags.csv')


# Defining our necessary functions 
Because for easy and quick production, I have build functions for every major part of data pre-processing and model building. In this steps, I store all those functions without storing them spreading all over the notebook. I provide rich documentation for each function so that they will be easily understandable. 

In [5]:
def generate_int_id(dataframe, id_col_name):
    """
    Generate unique integer id for users, questions and answers

    Parameters
    ----------
    dataframe: Dataframe
        Pandas Dataframe for Users or Q&A. 
    id_col_name : String 
        New integer id's column name.
        
    Returns
    -------
    Dataframe
        Updated dataframe containing new id column 
    """
    new_dataframe=dataframe.assign(
        int_id_col_name=np.arange(len(dataframe))
        ).reset_index(drop=True)
    return new_dataframe.rename(columns={'int_id_col_name': id_col_name})



def create_features(dataframe, features_name, id_col_name):
    """
    Generate features that will be ready for feeding into lightfm

    Parameters
    ----------
    dataframe: Dataframe
        Pandas Dataframe which contains features
    features_name : List
        List of feature columns name avaiable in dataframe
    id_col_name: String
        Column name which contains id of the question or
        answer that the features will map to.
        There are two possible values for this variable.
        1. questions_id_num
        2. professionals_id_num

    Returns
    -------
    Pandas Series
        A pandas series containing process features
        that are ready for feed into lightfm.
        The format of each value
        will be (user_id, ['feature_1', 'feature_2', 'feature_3'])
        Ex. -> (1, ['military', 'army', '5'])
    """

    features = dataframe[features_name].apply(
        lambda x: ','.join(x.map(str)), axis=1)
    features = features.str.split(',')
    features = list(zip(dataframe[id_col_name], features))
    return features



def generate_feature_list(dataframe, features_name):
    """
    Generate features list for mapping 

    Parameters
    ----------
    dataframe: Dataframe
        Pandas Dataframe for Users or Q&A. 
    features_name : List
        List of feature columns name avaiable in dataframe. 
        
    Returns
    -------
    List of all features for mapping 
    """
    features = dataframe[features_name].apply(
        lambda x: ','.join(x.map(str)), axis=1)
    features = features.str.split(',')
    features = features.apply(pd.Series).stack().reset_index(drop=True)
    return features


def calculate_auc_score(lightfm_model, interactions_matrix, 
                        question_features, professional_features): 
    """
    Measure the ROC AUC metric for a model. 
    A perfect score is 1.0.

    Parameters
    ----------
    lightfm_model: LightFM model 
        A fitted lightfm model 
    interactions_matrix : 
        A lightfm interactions matrix 
    question_features, professional_features: 
        Lightfm features 
        
    Returns
    -------
    String containing AUC score 
    """
    score = auc_score( 
        lightfm_model, interactions_matrix, 
        item_features=question_features, 
        user_features=professional_features, 
        num_threads=4).mean()
    return score

# Data Preprocessing and feature creation 
Data preprocessing is essential for every data science project. We need to clean and modified our data for our own usecases. Also, feature creation is very important. Because it's let's our model good and diverse prediction. 

**Generate numeric identifier**:
LightFM python only except numeric id. But the data CareerVillage has provided us is contains uuid for identifying users and professionals and others. In this step, I will make unique identifier for each professionals, students, questions and answers. 

In [6]:
# generating unique integer id for users and q&a
df_professionals = generate_int_id(df_professionals, 'professionals_id_num')
df_students = generate_int_id(df_students, 'students_id_num')
df_questions = generate_int_id(df_questions, 'questions_id_num')
df_answers = generate_int_id(df_answers, 'answers_id_num')

In [5]:
#  df_answers.groupby(['answers_author_id'], sort=False).ngroup()

**Merging Datasets**: This is one of the most important steps for our solution. Our professionals, students, q&a and tags are stored in seperate datasets. For purpose of model, we have to merge our datasets in very carefull way so that they are useful for our model. 

1. All tags (q&a) are stored in a separate dataset. So firstly we merge those tags with questions and answers datasets. 
2. Then, we merge answers with quesitons because one question can have multiple answers. 

In [7]:
###########################
# merging dataset
###########################

# just dropna from tags 
df_tags = df_tags.dropna()
df_tags['tags_tag_name'] = df_tags['tags_tag_name'].str.replace('#', '')


# merge tag_questions with tags name
# then group all tags for each question into single rows
df_tags_question = df_tag_questions.merge(
    df_tags, how='inner',
    left_on='tag_questions_tag_id', right_on='tags_tag_id')
df_tags_question = df_tags_question.groupby(
    ['tag_questions_question_id'])['tags_tag_name'].apply(
        ','.join).reset_index()
df_tags_question = df_tags_question.rename(columns={'tags_tag_name': 'questions_tag_name'})

# merge tag_users with tags name 
# then group all tags for each user into single rows 
# after that rename the tag column name 
df_tags_pro = df_tag_users.merge(
    df_tags, how='inner',
    left_on='tag_users_tag_id', right_on='tags_tag_id')
df_tags_pro = df_tags_pro.groupby(
    ['tag_users_user_id'])['tags_tag_name'].apply(
        ','.join).reset_index()
df_tags_pro = df_tags_pro.rename(columns={'tags_tag_name': 'professionals_tag_name'})


# merge professionals and questions tags with main merge_dataset 
df_questions = df_questions.merge(
    df_tags_question, how='left',
    left_on='questions_id', right_on='tag_questions_question_id')
df_professionals = df_professionals.merge(
    df_tags_pro, how='left',
    left_on='professionals_id', right_on='tag_users_user_id')

# merge questions with scores 
df_questions = df_questions.merge(
    df_question_scores, how='left',
    left_on='questions_id', right_on='id')
# merge questions with students 
df_questions = df_questions.merge(
    df_students, how='left',
    left_on='questions_author_id', right_on='students_id')



# merge answers with questions 
# then merge professionals and questions score with that 
df_merge = df_answers.merge(
    df_questions, how='inner',
    left_on='answers_question_id', right_on='questions_id')
df_merge = df_merge.merge(
    df_professionals, how='inner',
    left_on='answers_author_id', right_on='professionals_id')
df_merge = df_merge.merge(
    df_question_scores, how='inner',
    left_on='questions_id', right_on='id')

**Generate some features**: In this steps, we are going to generate some features. We are going to generate ```number of answers by professionals```, ```num of answers in each question```, ```num of tags per professionals``` and ```number of tags per question```. I will not use all of these features in this model. But I will use ```number of answers per question``` for weighting our model so that our model pay less attention to those quesitons that have higher number of answers. 

In [8]:
#######################
# Generate some features for calculates weights
# that will use with interaction matrix 
#######################

df_merge['num_of_ans_by_professional'] = df_merge.groupby(['answers_author_id'])['questions_id'].transform('count')
df_merge['num_ans_per_ques'] = df_merge.groupby(['questions_id'])['answers_id'].transform('count')
df_merge['num_tags_professional'] = df_merge['professionals_tag_name'].str.split(",").str.len()
df_merge['num_tags_question'] = df_merge['questions_tag_name'].str.split(",").str.len()



In [10]:
print("Maximum number of answer per question : " + str(df_merge['num_ans_per_ques'].max()))
print("Maximum number of tags per professional : " + str(df_merge['num_tags_professional'].max()))
print("Maximum number of tags per question : " + str(df_merge['num_tags_question'].max()))

Maximum number of answer per question : 58
Maximum number of tags per professional : 82.0
Maximum number of tags per question : 54.0


**Merge answered questions tags with professional's tags**: Professionals can follow some tags. But not all professional follow tags and most especially we see from EDA that sometime professionals answers questions that is not related to their tags. For that reason, I have merge questions tags that each professional has answered with professional tags. This makes our model more robust and context aware. 

In [11]:
########################
# Merge professionals previous answered 
# questions tags into professionals tags 
########################

# select professionals answered questions tags 
# and stored as a dataframe
professionals_prev_ans_tags = df_merge[['professionals_id', 'questions_tag_name']]
# drop null values from that 
professionals_prev_ans_tags = professionals_prev_ans_tags.dropna()
# because professsionals answers multiple questions, 
# we group all of tags of each user into single row 
professionals_prev_ans_tags = professionals_prev_ans_tags.groupby(
    ['professionals_id'])['questions_tag_name'].apply(
        ','.join).reset_index()

# drop duplicates tags from each professionals rows
professionals_prev_ans_tags['questions_tag_name'] = (
    professionals_prev_ans_tags['questions_tag_name'].str.split(',').apply(set).str.join(','))

# finally merge the dataframe with professionals dataframe 
df_professionals = df_professionals.merge(professionals_prev_ans_tags, how='left', on='professionals_id')

# join professionals tags and their answered tags 
# we replace nan values with ""
df_professionals['professional_all_tags'] = (
    df_professionals[['professionals_tag_name', 'questions_tag_name']].apply(
        lambda x: ','.join(x.dropna()),
        axis=1))


**Handling null and duplicates values**: Now we want clean our data a little bit. We will handle null and duplicate values. Because if we don't remove that they will cause error and wrong prediction. Also, we will replace null values with generic name or value.

In [12]:
# handling null values 
df_questions['score'] = df_questions['score'].fillna(0)
df_questions['score'] = df_questions['score'].astype(int)
df_questions['questions_tag_name'] = df_questions['questions_tag_name'].fillna('No Tag')
# remove duplicates tags from each questions 
df_questions['questions_tag_name'] = df_questions['questions_tag_name'].str.split(',').apply(set).str.join(',')


# fill nan with 'No Tag' if any 
df_professionals['professional_all_tags'] = df_professionals['professional_all_tags'].fillna('No Tag')
# replace "" with "No Tag", because previously we replace nan with ""
df_professionals['professional_all_tags'] = df_professionals['professional_all_tags'].replace('', 'No Tag')
df_professionals['professionals_location'] = df_professionals['professionals_location'].fillna('No Location')
df_professionals['professionals_industry'] = df_professionals['professionals_industry'].fillna('No Industry')

# remove duplicates tags from each professionals 
df_professionals['professional_all_tags'] = df_professionals['professional_all_tags'].str.split(',').apply(set).str.join(',')



# remove some null values from df_merge
df_merge['num_ans_per_ques']  = df_merge['num_ans_per_ques'].fillna(0)
df_merge['num_tags_professional'] = df_merge['num_tags_professional'].fillna(0)
df_merge['num_tags_question'] = df_merge['num_tags_question'].fillna(0)

# Building model in LightFM
In this steps, we are going to build our lighFM model using lightFM python library. Firstly, we have to create lightFM ```Dataset``` for our model. LightFM Datset class makes it really easy for us for creating ```interection matrix```, ```weights``` and ```user/item features```.
* ```interection matrix```: It is a matrix that contains user/ item interections or professional/quesiton intereactions. 
* ```weights```: weight of interection matrix. Less weight means less importance to that interection matrix. 
* ```user/item features```: user/item features supplied as like this ```(user_id, ['feature_1', 'feature_2', 'feature_3'])```

If you want to how lightfm python library's dataset class works and how to use it, please go to this link [Building LightFM Datasets](http://https://lyst.github.io/lightfm/docs/examples/dataset.html). 

Then, after that we will be start building our LightFM model using LightFM class. LightFM class makes it really easy for making lightFM model. After that we will train our model by our data. 

**Creating features list for Dataset class**: LightFM library has a Dataset class that makes it really easy for building necessary information for model. But we have feed set of all professionals/questions unique ids and all questions and professional features list. This will create internel mapping for lightFM to use. 

In [13]:
# generating features list for mapping 
question_feature_list = generate_feature_list(
    df_questions,
    ['questions_tag_name'])

professional_feature_list = generate_feature_list(
    df_professionals,
    ['professional_all_tags'])

In [14]:
# calculate our weight value 
df_merge['total_weights'] = 1 / (
    df_merge['num_ans_per_ques'])


# creating features for feeding into lightfm 
df_questions['question_features'] = create_features(
    df_questions, ['questions_tag_name'], 
    'questions_id_num')

df_professionals['professional_features'] = create_features(
    df_professionals,
    ['professional_all_tags'],
    'professionals_id_num')

**LightFM Dataset**: In this steps we are going to build lightfm datasets. And then we will be building our interactions matrix, weights and professional/question features. 

In [15]:
########################
# Dataset building for lightfm
########################

# define our dataset variable
# then we feed unique professionals and questions ids
# and item and professional feature list
# this will create lightfm internel mapping
dataset = Dataset()
dataset.fit(
    set(df_professionals['professionals_id_num']), 
    set(df_questions['questions_id_num']),
    item_features=question_feature_list, 
    user_features=professional_feature_list)


# now we are building interactions matrix between professionals and quesitons
# we are passing professional and questions id as a tuple
# e.g -> pd.Series((pro_id, question_id), (pro_id, questin_id))
# then we use lightfm build in method for building interactions matrix
df_merge['author_question_id_tuple'] = list(zip(
    df_merge.professionals_id_num, df_merge.questions_id_num, df_merge.total_weights))

interactions, weights = dataset.build_interactions(
    df_merge['author_question_id_tuple'])



# now we are building our questions and professionals features
# in a way that lightfm understand.
# we are using lightfm build in method for building
# questions and professionals features 
questions_features = dataset.build_item_features(
    df_questions['question_features'])

professional_features = dataset.build_user_features(
    df_professionals['professional_features'])

**Model building and training**: In ths steps, I am going to build lightfm model and then train the model. If you want to learn how to create lightfm model using this library please read this post [recommender for the Movielens dataset](https://lyst.github.io/lightfm/docs/examples/movielens_implicit.html). 

In [16]:
################################
# Model building part
################################

# define lightfm model by specifying hyper-parametre
# then fit the model with ineteractions matrix, item and user features 
model = LightFM(
    no_components=150,
    learning_rate=0.05,
    loss='warp',
    random_state=2019)

model.fit(
    interactions,
    item_features=questions_features,
    user_features=professional_features, sample_weight=weights,
    epochs=5, num_threads=4, verbose=True)


Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4


# Evaluating the performance of the model 
Now we have to evaluate our model to see it's performance. No matter how good your model is, if you can't evaluate your model correctly you can't imporove and trust your model. For recommendation problem, there is not very good matrics for evaluating. But luckily lightfm provides us a very rich set of evaluating matrics. In this steps, we will be calculating AUC scores for our model.

**What is AUC score in lightfm library?**: It measure the ROC AUC metric for a model: the probability that a randomly chosen positive example has a higher score than a randomly chosen negative example. A perfect score is 1.0. 

Let's see what is our model score. 

In [17]:
calculate_auc_score(model, interactions, questions_features, professional_features)

0.9139259

Wow! That is really impresive. Over AUC is over 90 percent. That is really excellent. This tells us that the quality of our overall model is very good.

**Make real recommendations**: Now we already see how our model is by looking at AUC score. But now let's see some real example of recommendation. 

In [18]:
from IPython.display import display_html
def display_side_by_side(*args):
    html_str=''
    for df in args:
        html_str+=df.to_html()
    display_html(html_str.replace('table','table style="display:inline"'),raw=True)

def recommend_questions(professional_ids):
     
    for professional in professional_ids:
        # print their previous answered question title
        previous_q_id_num = df_merge.loc[df_merge['professionals_id_num'] == professional][:3]['questions_id_num']
        df_previous_questions = df_questions.loc[df_questions['questions_id_num'].isin(previous_q_id_num)]
        print('Professional Id (' + str(professional) + "): Previous Answered Questions")
        display_side_by_side(
            df_previous_questions[['questions_title', 'question_features']],
            df_professionals.loc[df_professionals.professionals_id_num == professional][['professionals_id_num','professionals_tag_name']])
        
        # predict
        discard_qu_id = df_previous_questions['questions_id_num'].values.tolist()
        df_use_for_prediction = df_questions.loc[~df_questions['questions_id_num'].isin(discard_qu_id)]
        questions_id_for_predict = df_use_for_prediction['questions_id_num'].values.tolist()
        
        scores = model.predict(
            professional,
            questions_id_for_predict,
            item_features=questions_features,
            user_features=professional_features)
        
        df_use_for_prediction['scores'] = scores
        df_use_for_prediction = df_use_for_prediction.sort_values(by='scores', ascending=False)[:8]
        print('Professional Id (' + str(professional) + "): Recommended Questions: ")
        display(df_use_for_prediction[['questions_title', 'question_features']])
    

    

    

In [19]:
recommend_questions([1200 ,19897, 3])

Professional Id (1200): Previous Answered Questions


,questions_title,question_features
,professionals_id_num,professionals_tag_name
1200,1200,"marketing,strategy,entrepreneurship,management,java,advertising,python,data-analysis,online-advertising,real-estate,team-leadership,dj,analytics,display-advertising,football,blackjack,hip-hop,billiards,break"


Professional Id (1200): Recommended Questions: 


,questions_title,question_features
19011,How do you get started in starting your own bu...,"(19011, [business, management, marketing])"
7750,Are there any good colleges for learning bus...,"(7750, [business, marketing])"
23680,what makes markerting so important in business?,"(23680, [business, marketing])"
9802,Is marketing a good major?,"(9802, [business, marketing])"
22676,What are some specific daily responsibilities ...,"(22676, [business, marketing, marketing-and-ad..."
20431,What are the average income for someone that w...,"(20431, [business, entrepreneurship])"
15671,How beneficial will a business major be ?,"(15671, [accountant, marketing, accounting, bu..."
2357,Is It hard to start up and own a business?,"(2357, [business, entrepreneurship])"


Professional Id (19897): Previous Answered Questions


...

,questions_title,question_features
6058,How should you start in the Graphic Design ind...,"(6058, [design, art, graphic-design])"
19407,How can you be a successful photographer? What...,"(19407, [art, photography, graphic-design])"
2310,what is one of best things about being an anim...,"(2310, [artist, design, art, animation])"
23691,How to comopose/write music without going ti s...,"(23691, [music, art])"
9682,How to get started in animation?,"(9682, [artist, art, animation])"
13484,Would a Graphic Design degree be a feesible op...,"(13484, [art, graphic-design])"
17325,what are the required fields forgraphic design?,"(17325, [art, graphic-design])"
11231,Do I have to live in a big city to do well in ...,"(11231, [web-design, art, graphic-design, arti..."


Professional Id (3): Previous Answered Questions


,questions_title,question_features
11339,What are the different jobs a person can do in Forensic Science?,"(11339, [justice, forensic, science, criminal])"
14818,What does a typical work day for a forensic scientist look like?,"(14818, [No Tag])"
19077,Is most of your day spent working when being a detective?,"(19077, [detective])"
,professionals_id_num,professionals_tag_name
3,3,NaN


Professional Id (3): Recommended Questions: 


,questions_title,question_features
2423,How long does it take to become a Detective?,"(2423, [police, law, lawyer, criminal-justice,..."
17184,What types of Detectives are there?,"(17184, [police, law, lawyer, criminal-justice..."
9778,I want to be a police officer or a police disp...,"(9778, [police, law, criminal-justice, police-..."
16214,"Do you go to college, then B.L.E.T( Basic Law ...","(16214, [police, law, law-enforcement])"
18947,"Could I go straight into Law Enforcememt, when...","(18947, [police, law, law-enforcement])"
1941,What are important characteristics of a lawyer?,"(1941, [law-school, lawyer, law, law-enforceme..."
18124,what is the starting salary for a police offic...,"(18124, [law, law-enforcement])"
18872,What majors would fit a law enforcement career?,"(18872, [police, law, law-enforcement])"


**Analysis**: Awesome! Finally we can see our recommendation by our model. Let's take some time to ponder over the recommendations. 
* For the first professionl (1200) has not answer any questions yet. But he/she follows some tags. Our model take those tags as features and predict questions that has similar tags. 
* For the second professionals (19897) answered one questions that has tag major. But in his profile he follows tags like creative works like arts, illustrator etc. So our model recommend questions that has creative tags like arts, illustrator because he follows more tags one creative works. 
* For the third professionals (3): answered questions that has tag forensic, criminal, science, justice, detective. From the tags we can get an idea of professionals interests. Our model also learn that. That's why it recommend items that has tags like law, criminal, detective. 

This is just a simple exploration. Hope you get idea of the model recommendations. The model can survive cold-start, high-poularity problem. It also recommend those questions that has less answer because of its weights that I provided during training. Now we build our model and tested it. In the next section, we will look how we can put this model in production. 

# Model in Production
We previously saw how lightfm model works and build for this project. Well, now we are going build a pipeline that will help us for putting this model into production. We are going to build class for each steps discuss in step 2. Also, we are going to build some additional functions and methods that will add additional functionality to the model. 

Here is the picture of our pipeline: 
![](https://i.imgur.com/Kh4rVcL.png)


We will now build class for each of these steps. Without further do let's begin. 

In [21]:
############################################
# Read all our datasets agian 
# and store them in pandas dataframe objects. 
############################################
base_path = 'data/careervillage/'
df_answer_scores = pd.read_csv(
    base_path + 'answer_scores.csv')

df_answers = pd.read_csv(
    base_path + 'answers.csv',
    parse_dates=['answers_date_added'])

df_comments = pd.read_csv(
    base_path + 'comments.csv')

df_emails = pd.read_csv(
    base_path + 'emails.csv')

df_group_memberships = pd.read_csv(
    base_path + 'group_memberships.csv')

df_groups = pd.read_csv(
    base_path + 'groups.csv')

df_matches = pd.read_csv(
    base_path + 'matches.csv')

df_professionals = pd.read_csv(
    base_path + 'professionals.csv',
    parse_dates=['professionals_date_joined'])

df_question_scores = pd.read_csv(
    base_path + 'question_scores.csv')

df_questions = pd.read_csv(
    base_path + 'questions.csv',
    parse_dates=['questions_date_added'])

df_school_memberships = pd.read_csv(
    base_path + 'school_memberships.csv')

df_students = pd.read_csv(
    base_path + 'students.csv',
    parse_dates=['students_date_joined'])

df_tag_questions = pd.read_csv(
    base_path + 'tag_questions.csv')

df_tag_users = pd.read_csv(
    base_path + 'tag_users.csv')

df_tags = pd.read_csv(
    base_path + 'tags.csv')


**Data Processing Class**: Now we are going to build a class that will be used for data cleaning and processing specificly designed for CareerVillage Datasetes. I have provided details document and comment with each part of the code. This will help understanding the code and intention very well. 

In [23]:
class CareerVillageDataPreparation:
    """
    Clean and process data CareerVillage Data. 
    
    This class process data in a way that will be useful
    for building lightFM dataset. 
    """
    
    def __init__(self):
        pass

    def _assign_unique_id(self, data, id_col_name):
        """
        Generate unique integer id for users, questions and answers

        Parameters
        ----------
        data: Dataframe
            Pandas Dataframe for Users or Q&A. 
        id_col_name : String 
            New integer id's column name.

        Returns
        -------
        Dataframe
            Updated dataframe containing new id column
        """
        new_dataframe=data.assign(
            int_id_col_name=np.arange(len(data))
            ).reset_index(drop=True)
        return new_dataframe.rename(columns={'int_id_col_name': id_col_name})

    def _dropna(self, data, column, axis):
        """Drop null values from specific column"""
        return data.dropna(column, axis=axis)

    def _merge_data(self, left_data, left_key, right_data, right_key, how):
        """
        This function is used for merging two dataframe.
        
        Parameters
        -----------
        left_data: Dataframe
            Left side dataframe for merge
        left_key: String
            Left Dataframe merge key
        right_data: Dataframe
            Right side dataframe for merge
        right_key: String
            Right Dataframe merge key
        how: String
            Method of merge (inner, left, right, outer)
            
        
        Returns
        --------
        Dataframe
            A new dataframe merging left and right dataframe
        """
        return left_data.merge(
            right_data,
            how=how,
            left_on=left_key,
            right_on=right_key)

    def _group_tags(self, data, group_by, tag_column):
        """Grouop multiple tags into single rows sepearated by comma"""
        return data.groupby(
            [group_by])[tag_column].apply(
            ','.join).reset_index()

    def _merge_cv_datasets(
        self,
        professionals,students,
        questions,answers,
        tags,tag_questions,tag_users, questions_score):
        """
        This function merges all the necessary 
        CareerVillage dataset in defined way. 
        
        Parameters
        ------------
        professionals,students,
        questions,answers,
        tags,tag_questions,
        tag_users,
        questions_score: Dataframe
            Pandas dataframe defined by it's name
        
        
        Returns
        ---------
        questions, professionals: Dataframe
            Updated dataframe after merge
        merge: Dataframe
            A new datframe after merging answers with questions
        """
        
        
        # merge tag_questions with tags name
        # then group all tags for each question into single rows
        tag_question = self._merge_data(
            left_data=tag_questions,
            left_key='tag_questions_tag_id',
            right_data=tags,
            right_key='tags_tag_id',
            how='inner')
        tag_question = self._group_tags(
            data=tag_question,
            group_by='tag_questions_question_id',
            tag_column='tags_tag_name')
        
        tag_question = tag_question.rename(
            columns={'tags_tag_name': 'questions_tag_name'})
        
        # merge tag_users with tags name
        # then group all tags for each user into single rows 
        # after that rename the tag column name
        tags_pro = self._merge_data(
            left_data=tag_users,
            left_key='tag_users_tag_id',
            right_data=tags,
            right_key='tags_tag_id',
            how='inner')
        tags_pro = self._group_tags(
            data=tags_pro,
            group_by='tag_users_user_id',
            tag_column='tags_tag_name')
        tags_pro = tags_pro.rename(
            columns={'tags_tag_name': 'professionals_tag_name'})
        
        # merge professionals and questions tags with main merge_dataset 
        questions = self._merge_data(
            left_data=questions,
            left_key='questions_id',
            right_data=tag_question,
            right_key='tag_questions_question_id',
            how='left')
        professionals = self._merge_data(
            left_data=professionals,
            left_key='professionals_id',
            right_data=tags_pro,
            right_key='tag_users_user_id',
            how='left')
        
        # merge questions with scores 
        questions = self._merge_data(
            left_data=questions,
            left_key='questions_id',
            right_data=questions_score,
            right_key='id',
            how='left')
        
        # merge questions with students
        questions = self._merge_data(
            left_data=questions,
            left_key='questions_author_id',
            right_data=students,
            right_key='students_id',
            how='left')
        
        # merge answers with questions
        # then merge professionals and questions score with that
        merge = self._merge_data(
            left_data=answers,
            left_key='answers_question_id',
            right_data=questions,
            right_key='questions_id',
            how='inner')
        
        merge = self._merge_data(
            left_data=merge,
            left_key='answers_author_id',
            right_data=professionals,
            right_key='professionals_id',
            how='inner')
        
        return questions, professionals, merge
  
    def _drop_duplicates_tags(self, data, col_name):
        # drop duplicates tags from each row
        return (
            data[col_name].str.split(
                ',').apply(set).str.join(','))


    def _merge_pro_pre_ans_tags(self, professionals, merge):
        ########################
        # Merge professionals previous answered
        # questions tags into professionals tags
        ########################
        
        # select professionals answered questions tags
        # and stored as a dataframe
        professionals_prev_ans_tags = (
            merge[['professionals_id', 'questions_tag_name']])
        # drop null values from that
        professionals_prev_ans_tags = professionals_prev_ans_tags.dropna()
        
        # because professsionals answers multiple questions,
        # we group all of tags of each user into single row
        professionals_prev_ans_tags = self._group_tags(
            data=professionals_prev_ans_tags,
            group_by='professionals_id',
            tag_column='questions_tag_name')
        
        # drop duplicates tags from each professionals rows
        professionals_prev_ans_tags['questions_tag_name'] = \
        self._drop_duplicates_tags(
            professionals_prev_ans_tags, 'questions_tag_name')
        
        # finally merge the dataframe with professionals dataframe
        professionals = self._merge_data(
            left_data=professionals,
            left_key='professionals_id',
            right_data=professionals_prev_ans_tags,
            right_key='professionals_id',
            how='left')
        
        # join professionals tags and their answered tags 
        # we replace nan values with ""
        professionals['professional_all_tags'] = (
            professionals[['professionals_tag_name',
                           'questions_tag_name']].apply(
                lambda x: ','.join(x.dropna()),
                axis=1))
        return professionals

    def prepare(
        self,
        professionals,students,
        questions,answers,
        tags,tag_questions,tag_users, questions_score):
        
        """
        This function clean and process 
        CareerVillage Data sets. 
        """
        
        # assign unique integer id
        professionals = self._assign_unique_id(
            professionals, 'professionals_id_num')
        students = self._assign_unique_id(
            students, 'students_id_num')
        questions = self._assign_unique_id(
            questions, 'questions_id_num')
        answers = self._assign_unique_id(
            answers, 'answers_id_num')
        
        # just dropna from tags 
        tags = tags.dropna()
        tags['tags_tag_name'] = tags['tags_tag_name'].str.replace(
            '#', '')
        
        
        # merge necessary datasets
        df_questions, df_professionals, df_merge = self._merge_cv_datasets(
            professionals,students,
            questions,answers,
            tags,tag_questions,tag_users,
            questions_score)
        
        #######################
        # Generate some features for calculates weights
        # that will use with interaction matrix
        #######################
        df_merge['num_ans_per_ques'] = df_merge.groupby(
            ['questions_id'])['answers_id'].transform('count')
        
        # merge pro previoius answered question tags with pro tags 
        df_professionals = self._merge_pro_pre_ans_tags(
            df_professionals, df_merge)
        
        # some more pre-processing 
        # handling null values 
        df_questions['score'] = df_questions['score'].fillna(0)
        df_questions['score'] = df_questions['score'].astype(int)
        df_questions['questions_tag_name'] = \
        df_questions['questions_tag_name'].fillna('No Tag')
        
        # remove duplicates tags from each questions 
        df_questions['questions_tag_name'] = \
        df_questions['questions_tag_name'].str.split(
            ',').apply(set).str.join(',')

        # fill nan with 'No Tag' if any 
        df_professionals['professional_all_tags'] = \
        df_professionals['professional_all_tags'].fillna(
            'No Tag')
        # replace "" with "No Tag", because previously we replace nan with ""
        df_professionals['professional_all_tags'] = \
        df_professionals['professional_all_tags'].replace(
            '', 'No Tag')
        
        df_professionals['professionals_location'] = \
        df_professionals['professionals_location'].fillna(
            'No Location')
        
        df_professionals['professionals_industry'] = \
        df_professionals['professionals_industry'].fillna(
            'No Industry')

        # remove duplicates tags from each professionals
        df_professionals['professional_all_tags'] = \
        df_professionals['professional_all_tags'].str.split(
            ',').apply(set).str.join(',')

        # remove some null values from df_merge
        df_merge['num_ans_per_ques']  = \
        df_merge['num_ans_per_ques'].fillna(0)
        
        return df_questions, df_professionals, df_merge


**Building Data for LightFM Class**: From step 2 we already know that lightfm library except data in a very specific and elligent way. LightFM data format is already discussed in step 2. Feel free to read that. Now we are building a class that will be put all of dataset building puzzle in a specific class. 

In [24]:
class LightFMDataPrep:
    def __init__(self):
        pass
    def create_features(self, dataframe, features_name, id_col_name):
        """
        Generate features that will be ready for feeding into lightfm

        Parameters
        ----------
        dataframe: Dataframe
            Pandas Dataframe which contains features
        features_name : List
            List of feature columns name avaiable in dataframe
        id_col_name: String
            Column name which contains id of the question or
            answer that the features will map to.
            There are two possible values for this variable.
            1. questions_id_num
            2. professionals_id_num

        Returns
        -------
        Pandas Series
            A pandas series containing process features
            that are ready for feed into lightfm.
            The format of each value
            will be (user_id, ['feature_1', 'feature_2', 'feature_3'])
            Ex. -> (1, ['military', 'army', '5'])
        """

        features = dataframe[features_name].apply(
            lambda x: ','.join(x.map(str)), axis=1)
        features = features.str.split(',')
        features = list(zip(dataframe[id_col_name], features))
        return features



    def generate_feature_list(self, dataframe, features_name):
        """
        Generate features list for mapping 

        Parameters
        ----------
        dataframe: Dataframe
            Pandas Dataframe for Users or Q&A. 
        features_name : List
            List of feature columns name avaiable in dataframe. 

        Returns
        -------
        List of all features for mapping 
        """
        features = dataframe[features_name].apply(
            lambda x: ','.join(x.map(str)), axis=1)
        features = features.str.split(',')
        features = features.apply(pd.Series).stack().reset_index(drop=True)
        return features
    
    def create_data(self, questions, professionals, merge):
        question_feature_list = self.generate_feature_list(
            questions,
            ['questions_tag_name'])

        professional_feature_list = self.generate_feature_list(
            professionals,
            ['professional_all_tags'])
        
        merge['total_weights'] = 1 / (
            merge['num_ans_per_ques'])
        
        # creating features for feeding into lightfm 
        questions['question_features'] = self.create_features(
            questions, ['questions_tag_name'], 
            'questions_id_num')

        professionals['professional_features'] = self.create_features(
            professionals,
            ['professional_all_tags'],
            'professionals_id_num')
        
        return question_feature_list,\
    professional_feature_list,merge,questions,professionals
        
    def fit(self, questions, professionals, merge):
        ########################
        # Dataset building for lightfm
        ########################
        question_feature_list, \
        professional_feature_list,\
        merge,questions,professionals = \
        self.create_data(questions, professionals, merge)
        
        
        # define our dataset variable
        # then we feed unique professionals and questions ids
        # and item and professional feature list
        # this will create lightfm internel mapping
        dataset = Dataset()
        dataset.fit(
            set(professionals['professionals_id_num']), 
            set(questions['questions_id_num']),
            item_features=question_feature_list, 
            user_features=professional_feature_list)


        # now we are building interactions
        # matrix between professionals and quesitons
        # we are passing professional and questions id as a tuple
        # e.g -> pd.Series((pro_id, question_id), (pro_id, questin_id))
        # then we use lightfm build in method for building interactions matrix
        merge['author_question_id_tuple'] = list(zip(
            merge.professionals_id_num,
            merge.questions_id_num,
            merge.total_weights))

        interactions, weights = dataset.build_interactions(
            merge['author_question_id_tuple'])



        # now we are building our questions and
        # professionals features
        # in a way that lightfm understand.
        # we are using lightfm build in method for building
        # questions and professionals features 
        questions_features = dataset.build_item_features(
            questions['question_features'])

        professional_features = dataset.build_user_features(
            professionals['professional_features'])
        
        return interactions,\
    weights,questions_features,professional_features
        
        

**Train Model Class**: In step 2, we saw how we build and train our model. Now we are going to put those all together in TrainLightFM class. 

In [25]:
class TrainLightFM:
    def __init__(self):
        pass
        
    def train_test_split(self, interactions, weights):
        train_interactions, test_interactions = \
        cross_validation.random_train_test_split(
            interactions, 
            random_state=np.random.RandomState(2019))
        
        train_weights, test_weights = \
        cross_validation.random_train_test_split(
            weights, 
            random_state=np.random.RandomState(2019))
        return train_interactions,\
    test_interactions, train_weights, test_weights
    
    def fit(self, interactions, weights,
            questions_features, professional_features,
            cross_validation=False,no_components=150,
            learning_rate=0.05,
            loss='warp',
            random_state=2019,
            verbose=True,
            num_threads=4, epochs=5):
        ################################
        # Model building part
        ################################

        # define lightfm model by specifying hyper-parametre
        # then fit the model with ineteractions matrix,
        # item and user features
        
        model = LightFM(
            no_components,
            learning_rate,
            loss=loss,
            random_state=random_state)
        model.fit(
            interactions,
            item_features=questions_features,
            user_features=professional_features, sample_weight=weights,
            epochs=epochs, num_threads=num_threads, verbose=verbose)
        
        return model


**Recommendations classs**: Now we are going to build a class for making recommendations. This will make easy for making recommendations in djono api. This recommendations class build with extra features. You can use this for general prediction by giving professionals ids and questions features. It has another features that let's choose questions from range of two dates and make recommendation from those questions. 

This is useful because those professionals that choose email frequency lavel as "weekly" or "daily", we can select questions from a week and then recommend those questions. 

In [26]:
class LightFMRecommendations:
    """
    Make prediction given model and professional ids
    """
    def __init__(self, lightfm_model,
                 professionals_features,
                 questions_features,
                 questions,professionals,merge):
        self.model = lightfm_model
        self.professionals_features = professionals_features
        self.questions_features = questions_features
        self.questions = questions
        self.professionals = professionals
        self.merge = merge
        
    def previous_answered_questions(self, professionals_id):
        previous_q_id_num = (
            self.merge.loc[\
                self.merge['professionals_id_num'] == \
                professionals_id]['questions_id_num'])
        
        previous_answered_questions = self.questions.loc[\
            self.questions['questions_id_num'].isin(
            previous_q_id_num)]
        return previous_answered_questions
        
    
    def _filter_question_by_pro(self, professionals_id):
        """Drop questions that professional already answer"""
        previous_answered_questions = \
        self.previous_answered_questions(professionals_id)
        
        discard_qu_id = \
        previous_answered_questions['questions_id_num'].values.tolist()
        
        questions_for_prediction = \
        self.questions.loc[~self.questions['questions_id_num'].isin(discard_qu_id)]
        
        return questions_for_prediction
    
    def _filter_question_by_date(self, questions, start_date, end_date):
        mask = \
        (questions['questions_date_added'] > start_date) & \
        (questions['questions_date_added'] <= end_date)
        
        return questions.loc[mask]
        
    
    def recommend_by_pro_id_general(self,
                                    professional_id,
                                    num_prediction=8):
        questions_for_prediction = self._filter_question_by_pro(professional_id)
        score = self.model.predict(
            professional_id,
            questions_for_prediction['questions_id_num'].values.tolist(), 
            item_features=self.questions_features,
            user_features=self.professionals_features)
        
        questions_for_prediction['recommendation_score'] = score
        questions_for_prediction = questions_for_prediction.sort_values(
            by='recommendation_score', ascending=False)[:num_prediction]
        return questions_for_prediction
    
    def recommend_by_pro_id_frequency_date_range(self,
                                                 professional_id,
                                                 start_date,
                                                 end_date,
                                                 num_prediction=8):
        questions_for_prediction = \
        self._filter_question_by_pro(professional_id)
        
        start_date = datetime.strptime(start_date, '%Y-%m-%d')
        end_date = datetime.strptime(end_date, '%Y-%m-%d')
        
        questions_for_prediction = self._filter_question_by_date(
            questions_for_prediction, start_date, end_date)
        
        score = self.model.predict(
            professional_id,
            questions_for_prediction['questions_id_num'].values.tolist(), 
            item_features=self.questions_features,
            user_features=self.professionals_features)
        
        questions_for_prediction['recommendation_score'] = score
        questions_for_prediction = questions_for_prediction.sort_values(
            by='recommendation_score', ascending=False)[:num_prediction]
        return questions_for_prediction


**Put it all together**: Now we defined all our important class file. Let's use each of these class and build our model. 

In [27]:
# instiate all class instance
cv_data_prep = CareerVillageDataPreparation()
light_fm_data_prep = LightFMDataPrep()
train_lightfm = TrainLightFM()

# process raw data
df_questions_p, df_professionals_p, df_merge_p = \
cv_data_prep.prepare(
    df_professionals,df_students,
    df_questions,df_answers,
    df_tags,df_tag_questions,df_tag_users,
    df_question_scores)


# prepare data for lightfm 
interactions, weights, \
questions_features, professional_features = \
light_fm_data_prep.fit(
    df_questions_p, df_professionals_p, df_merge_p)


# finally build and trian our model
model = train_lightfm.fit(interactions,
                          weights,
                          questions_features,
                          professional_features)


Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4


Awesome! Do you see, how easy it was for building our model. We can surely apply this idea when putting the model into production. Now we are going to see some real recommendations. 

In [28]:
# define our recommender class
lightfm_recommendations = LightFMRecommendations(
    model,
    professional_features,questions_features,
    df_questions_p, df_professionals_p, df_merge_p)

# let's what our model predict for user id 3
print("Recommendation for professional: " + str(3))
display(lightfm_recommendations.recommend_by_pro_id_general(3)[:8])

Recommendation for professional: 3


,questions_id,questions_author_id,questions_date_added,questions_title,questions_body,questions_id_num,tag_questions_question_id,questions_tag_name,id,score,students_id,students_location,students_date_joined,students_id_num,question_features,recommendation_score
2423,9515b833b2ac4092a8b1a8cdb380781f,941ae126a59745fa9b4556293b38c1fb,2019-01-08 20:47:44+00:00,How long does it take to become a Detective?,#law #criminal-justice #lawyer #police #law-en...,2423,9515b833b2ac4092a8b1a8cdb380781f,"police,law,lawyer,criminal-justice,law-enforce...",9515b833b2ac4092a8b1a8cdb380781f,2,941ae126a59745fa9b4556293b38c1fb,"Oakland, California",2019-01-08 20:35:58+00:00,30755.0,"(2423, [police, law, lawyer, criminal-justice,...",-2.135185
17184,570ca25a625d461abffac230ea110db5,941ae126a59745fa9b4556293b38c1fb,2019-01-10 01:48:47+00:00,What types of Detectives are there?,#law #criminal-justice #lawyer #law-enforcemen...,17184,570ca25a625d461abffac230ea110db5,"police,law,lawyer,criminal-justice,law-enforce...",570ca25a625d461abffac230ea110db5,2,941ae126a59745fa9b4556293b38c1fb,"Oakland, California",2019-01-08 20:35:58+00:00,30755.0,"(17184, [police, law, lawyer, criminal-justice...",-2.165288
9778,776e22d9eb1045eb8a9771eb015e8ddf,d7601a6cc1d04e61aaa16c95cbd0b128,2018-10-03 14:04:13+00:00,I want to be a police officer or a police disp...,#police-officer #law #law-enforcement #crimina...,9778,776e22d9eb1045eb8a9771eb015e8ddf,"police,law,criminal-justice,police-officer,law...",776e22d9eb1045eb8a9771eb015e8ddf,2,d7601a6cc1d04e61aaa16c95cbd0b128,"Olney, Illinois",2018-10-03 14:01:25+00:00,29951.0,"(9778, [police, law, criminal-justice, police-...",-2.507841
10534,322afc67ac1848188a4a6d2bf5c51b20,8a8305d32bd144d5877842dcabdfb6d7,2016-05-05 15:42:36+00:00,Is there any required college courses to becom...,I am an explorer and is trying to set my caree...,10534,322afc67ac1848188a4a6d2bf5c51b20,"police,law,law-enforcement",322afc67ac1848188a4a6d2bf5c51b20,2,8a8305d32bd144d5877842dcabdfb6d7,"Laurinburg, North Carolina",2016-05-02 16:37:52+00:00,7103.0,"(10534, [police, law, law-enforcement])",-2.538090
18872,c3e6e57cb27b4134be9b8608a711e2fc,43f813594dd44e16843ecae4e2362ead,2015-03-23 21:17:34+00:00,What majors would fit a law enforcement career?,Im asking this question because I've heard tha...,18872,c3e6e57cb27b4134be9b8608a711e2fc,"police,law,law-enforcement",c3e6e57cb27b4134be9b8608a711e2fc,4,43f813594dd44e16843ecae4e2362ead,"Los Angeles, California",2015-03-23 21:09:01+00:00,3322.0,"(18872, [police, law, law-enforcement])",-2.551313
16214,ccb15b06a96a4bcfb4d5844550af25cc,8a8305d32bd144d5877842dcabdfb6d7,2016-05-04 16:32:58+00:00,"Do you go to college, then B.L.E.T( Basic Law ...",I am an explorer and is trying to set my caree...,16214,ccb15b06a96a4bcfb4d5844550af25cc,"police,law,law-enforcement",ccb15b06a96a4bcfb4d5844550af25cc,2,8a8305d32bd144d5877842dcabdfb6d7,"Laurinburg, North Carolina",2016-05-02 16:37:52+00:00,7103.0,"(16214, [police, law, law-enforcement])",-2.572512
1941,716e1eb45ae64de29633eacf2ebddc0e,0a49a80de472412988aac14f93b06374,2016-07-22 03:52:32+00:00,What are important characteristics of a lawyer?,I was curious about the desirable traits of a ...,1941,716e1eb45ae64de29633eacf2ebddc0e,"law-school,lawyer,law,law-enforcement",716e1eb45ae64de29633eacf2ebddc0e,3,0a49a80de472412988aac14f93b06374,"Plano, Texas",2016-05-30 21:08:55+00:00,11780.0,"(1941, [law-school, lawyer, law, law-enforceme...",-2.585965
18126,6c0079d59ae74c1388045fecbe585570,8a8305d32bd144d5877842dcabdfb6d7,2016-05-04 16:34:56+00:00,Do you need law enforcement background such as...,I am an explorer and is trying to set my caree...,18126,6c0079d59ae74c1388045fecbe585570,"police,law,law-enforcement",6c0079d59ae74c1388045fecbe585570,4,8a8305d32bd144d5877842dcabdfb6d7,"Laurinburg, North Carolina",2016-05-02 16:37:52+00:00,7103.0,"(18126, [police, law, law-enforcement])",-2.616544


In [29]:
# also let's see what our model predicts for professional 3
# given questions between two dates
print("Recommendations for professionals (question from 2016-1-1 to 2016-12-31): " + str(3))
display(lightfm_recommendations.recommend_by_pro_id_frequency_date_range(3,
                                                                 '2016-1-1','2016-12-31')[:8])

Recommendations for professionals (question from 2016-1-1 to 2016-12-31): 3


TypeError: Cannot compare tz-naive and tz-aware datetime-like objects

Awesome! We can see our recommendations. Also, we can see, the new recommendation class has a method for recommending questions by a frequency of date. This is very helpful for recommending questions to professionals that have set their email frequency to daily or weekly. 

# Conclusion

**Idea that I tried but don't implemented in this notebook**: 
* Adding location features: I tried adding location features but somehow it decreases model AUC score to 91% to 84%. That's why I don't use that features.
* Adding dates and hearts data: I also tried that but it doesn't improve AUC score. 
* Correcting spelling error: I tried this method and successfully implemented it. But this is really slow. For that reason, I exluded it. 

**Idea that I think is important don't implemented in this notebook**: 
* Adding professionals industry and title as a features. This will inhance our model diversity and will increase overall recommendations score.
* CareerVillage should auto correct the hashtags for students questions asking time. This will help the model to match the tags more efficiently. 
* For those professionals those have choosen email frequency to immediete, we can create another same model just exchange user/item features. I mean train our model by giving questions as users and professionals as items. In this way, we can predict professionals by giving a questions. So that it helps to target daily frequency professionals.

Finally we came to end. I want give you a big thank you for reading this notebook. I have provided a very good recommender system for CareerVillage in the notebook. If you find any mistakes or have any suggestions feel free to comment. And don't forget to upvote. Good luck! 

References: 

[1] [Improving Pairwise Learning for Item Recommendation from Implicit Feedback](http://webia.lip6.fr/~gallinar/gallinari/uploads/Teaching/WSDM2014-rendle.pdf)

[2] [Content-based filtering](https://towardsdatascience.com/what-are-product-recommendation-engines-and-the-various-versions-of-them-9dcab4ee26d5)

[3] [What Is Content-Based Filtering?](https://www.upwork.com/hiring/data/what-is-content-based-filtering/)

[4] [What Is Collaborative Filtering?](https://www.upwork.com/hiring/data/how-collaborative-filtering-works/)

[5] [Collaborative filtering](https://en.wikipedia.org/wiki/Collaborative_filtering)

[6] [LightFM model documentation](https://lyst.github.io/lightfm/docs/lightfm.html)

[7] [Metadata Embeddings for User and Item Cold-start Recommendations](https://arxiv.org/pdf/1507.08439.pdf)